# 02. Descarga, extracción y preprocesamiento de imágenes DBT

**Proyecto de clasificación binaria de imágenes de tomosíntesis digital de mama (BCS-DBT).**

Este cuaderno continúa `01_seleccion_y_particion.ipynb`: lee sus **seis CSV**, obtiene cada serie DICOM desde TCIA, extrae los cortes especificados y guarda imágenes `float32` de 512 × 512 y sus etiquetas en archivos `.npy`.

**Seguridad de ejecución:** por defecto la descarga está **desactivada** (`EJECUTAR_DESCARGA = False`). Cambiarla a `True` solo cuando se quiera descargar. `MODO_PRUEBA = True` limita la operación a una serie. Una prueba no demuestra que el procesamiento completo funcione con todos los volúmenes.

**Alcance:** implementación de referencia para reproducir el *procedimiento*; no implica que se hayan vuelto a procesar las series exactas de los entrenamientos originales ni que los resultados publicados sean reproducibles sin el resto del código y sus condiciones.

Referencia del servicio de descarga: [TCIA Utils](https://github.com/kirbyju/tcia_utils).

## 1. Dependencias

In [ ]:
%pip install -q tcia_utils pydicom opencv-python-headless pandas numpy

import ast
import re
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
from tcia_utils import nbia


## 2. Cargar los CSV generados por el notebook 1

En Colab se solicitarán **los seis archivos CSV individuales** (no un ZIP). Si se ejecuta localmente, se pueden colocar junto al cuaderno y omitir la carga interactiva.


In [ ]:
NOMBRES = [
    'balanceado_train', 'balanceado_val', 'balanceado_test',
    'desbalanceado_train', 'desbalanceado_val', 'desbalanceado_test'
]

FALTANTES = [f'{nombre}.csv' for nombre in NOMBRES if not Path(f'{nombre}.csv').exists()]
if FALTANTES:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(f'Faltan archivos: {FALTANTES}') from exc
    print('Seleccionar los CSV faltantes:', FALTANTES)
    files.upload()

faltantes = [f'{nombre}.csv' for nombre in NOMBRES if not Path(f'{nombre}.csv').exists()]
if faltantes:
    raise FileNotFoundError(f'Faltan CSV necesarios: {faltantes}')

conjuntos = {nombre: pd.read_csv(f'{nombre}.csv') for nombre in NOMBRES}
for nombre, df in conjuntos.items():
    print(f'{nombre}: {len(df)} series')
    print('  Columnas:', list(df.columns))


## 3. Interpretar las posiciones y verificar los tamaños planificados

In [ ]:
def leer_posiciones(valor):
    if isinstance(valor, (list, tuple, np.ndarray)):
        posiciones = list(valor)
    elif pd.isna(valor):
        raise ValueError('La lista de posiciones está vacía')
    else:
        posiciones = ast.literal_eval(str(valor))
    if not isinstance(posiciones, (list, tuple)) or not posiciones:
        raise ValueError(f'Posiciones inválidas: {valor!r}')
    posiciones = [float(p) for p in posiciones]
    if any(p < 0 or p > 1 for p in posiciones):
        raise ValueError(f'Posiciones fuera de [0,1]: {posiciones}')
    return posiciones

for nombre, df in conjuntos.items():
    if nombre.startswith('balanceado'):
        if 'posiciones' not in df:
            raise KeyError(f'{nombre} no contiene columna posiciones')
        df['posiciones'] = df['posiciones'].apply(leer_posiciones)
    else:
        df['posiciones'] = [[0.5] for _ in range(len(df))]
    print(f'{nombre}: {sum(map(len, df["posiciones"]))} imágenes previstas')

esperados = {'balanceado_train':3197, 'balanceado_val':400, 'balanceado_test':400,
             'desbalanceado_train':2904, 'desbalanceado_val':334, 'desbalanceado_test':334}
for nombre, df in conjuntos.items():
    assert sum(map(len, df['posiciones'])) == esperados[nombre], f'Tamaño inesperado: {nombre}'


## 4. Identificar las series en TCIA

La descarga exige el identificador **SeriesInstanceUID**. Si el CSV incluye esa columna se usa directamente; de lo contrario, se intenta recuperarlo de `classic_path`/`descriptive_path` *solo si el identificador aparece explícitamente en la ruta*. Si no aparece, hay que incorporar el UID mediante los metadatos de TCIA: **`StudyUID` no equivale a `SeriesInstanceUID`**. El notebook se detendrá con un mensaje en lugar de descargar la serie equivocada.

In [ ]:
UID_RE = re.compile(r'(?<![0-9.])(?:[0-9]+\.)+[0-9]+(?![0-9.])')

def uid_de_fila(fila):
    for col in ('SeriesInstanceUID', 'SeriesUID', 'series_uid', 'SeriesInstanceUid'):
        if col in fila.index and pd.notna(fila[col]):
            uid = str(fila[col]).strip()
            if UID_RE.fullmatch(uid):
                return uid
            raise ValueError(f'{col} no es un UID DICOM válido: {uid!r}')
    # La última parte de classic_path en algunos manifiestos corresponde al UID de serie.
    for col in ('classic_path', 'descriptive_path'):
        if col in fila.index and pd.notna(fila[col]):
            partes = re.split(r'[/\\]', str(fila[col]).strip('/\\'))
            for parte in reversed(partes):
                if UID_RE.fullmatch(parte):
                    return parte
    raise ValueError('No hay SeriesInstanceUID identificable. Añadir esa columna al CSV desde metadatos TCIA.')

# Diagnóstico: no falla el notebook completo si la columna de UID todavía debe incorporarse.
try:
    ejemplo_uid = uid_de_fila(conjuntos['balanceado_train'].iloc[0])
    print('Ejemplo de UID de serie:', ejemplo_uid)
except ValueError as exc:
    print('REVISAR METADATOS ANTES DE DESCARGAR:', exc)


## 5. Lectura y preprocesamiento de un volumen DICOM

In [ ]:
def procesar_cortes_dicom(ruta_dicom, posiciones):
    """Lee un DICOM multiframe (planos, filas, columnas) y produce (N,512,512)."""
    ds = pydicom.dcmread(str(ruta_dicom))
    volumen = ds.pixel_array
    if volumen.ndim != 3:
        raise ValueError(f'Se espera DICOM multiframe (N,H,W); forma recibida: {volumen.shape}')
    if int(getattr(ds, 'NumberOfFrames', volumen.shape[0])) != volumen.shape[0]:
        raise ValueError('Dimensión de cortes incoherente con NumberOfFrames')
    n = volumen.shape[0]
    imagenes = []
    for posicion in posiciones:
        if not (0 <= float(posicion) <= 1):
            raise ValueError(f'Posición fuera de rango: {posicion}')
        idx = int(float(posicion) * (n - 1))
        corte = volumen[idx].astype(np.float32)
        maximo = float(np.max(corte))
        if maximo > 0:
            corte = corte / maximo
        corte = cv2.resize(corte, (512, 512), interpolation=cv2.INTER_AREA)
        imagenes.append(corte.astype(np.float32))
    return np.stack(imagenes)


def identificar_dicom_multiframe(directorio, uid_esperado):
    """Localiza el volumen multiframe correcto dentro de una serie descargada."""
    candidatos = []
    for ruta in Path(directorio).rglob('*'):
        if not ruta.is_file():
            continue
        try:
            ds = pydicom.dcmread(str(ruta), stop_before_pixels=True)
        except Exception:
            continue
        if str(getattr(ds, 'SeriesInstanceUID', '')) != str(uid_esperado):
            continue
        if int(getattr(ds, 'NumberOfFrames', 1)) > 1:
            candidatos.append(ruta)
    if len(candidatos) != 1:
        raise RuntimeError(
            f'Se esperó un DICOM multiframe para la serie {uid_esperado}; '
            f'se encontraron {len(candidatos)}. Revisar estructura de los DICOM descargados.'
        )
    return candidatos[0]


## 6. Descargar una serie desde TCIA

`nbia.downloadSeries` recibe **una lista** de UIDs con `input_type="list"`; pasar una cadena directamente puede producir errores. Se descargan únicamente las series solicitadas. El archivo DICOM se localiza verificando su UID, en lugar de suponer un nombre fijo.

In [ ]:
CARPETA_DICOM = Path('dicom_temporales')
CARPETA_RESULTADOS = Path('resultados_npy')


def descargar_y_procesar_serie(uid, posiciones):
    destino = CARPETA_DICOM / uid
    destino.mkdir(parents=True, exist_ok=True)
    nbia.downloadSeries([uid], input_type='list', path=str(destino))
    dicom = identificar_dicom_multiframe(destino, uid)
    return procesar_cortes_dicom(dicom, posiciones)


## 7. Etiquetas y procesamiento por conjunto

Las imágenes y etiquetas se escriben en `.npy` mediante un archivo temporal de memoria mapeada, para no mantener el conjunto entero en RAM. Se guarda también un `manifest_*.csv` con la correspondencia entre filas y serie/corte. Ante un error, el conjunto no se marca como completo.

**Nota:** los CSV deben contar con la columna `clase` (0 = `Normal`, 1 = `Actionable`/`Benign`/`Cancer`).

In [ ]:
def etiqueta_de_fila(fila):
    for col in ('clase', 'label', 'etiqueta', 'target'):
        if col in fila.index and pd.notna(fila[col]):
            v = fila[col]
            if isinstance(v, str):
                v = v.strip().lower()
                if v in ('positiva', 'positivo', 'positive', 'pos', '1'):
                    return 1
                if v in ('negativa', 'negativo', 'negative', 'neg', '0'):
                    return 0
            if v in (0, 1):
                return int(v)
            raise ValueError(f'Etiqueta no reconocida en {col}: {v!r}')
    if all(c in fila.index for c in ('Normal', 'Actionable', 'Benign', 'Cancer')):
        return int(any(int(fila[c]) == 1 for c in ('Actionable','Benign','Cancer')))
    raise KeyError('No se pudo identificar la etiqueta: falta columna clase o banderas diagnósticas')


def procesar_conjunto(nombre, df, max_series=None, eliminar_dicom=False):
    df = df.iloc[:max_series] if max_series is not None else df
    if df.empty:
        raise ValueError('Conjunto vacío')

    CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)
    total = sum(len(p) for p in df['posiciones'])
    sufijo = f'{nombre}_prueba' if max_series is not None else nombre
    ruta_x = CARPETA_RESULTADOS / f'X_{sufijo}.npy'
    ruta_y = CARPETA_RESULTADOS / f'y_{sufijo}.npy'
    tmp_x = CARPETA_RESULTADOS / f'X_{sufijo}.parcial.npy'
    tmp_y = CARPETA_RESULTADOS / f'y_{sufijo}.parcial.npy'
    x = np.lib.format.open_memmap(tmp_x, mode='w+', dtype=np.float32,
                                  shape=(total, 512, 512))
    y = np.lib.format.open_memmap(tmp_y, mode='w+', dtype=np.int64, shape=(total,))
    registros = []
    cursor = 0
    try:
        for i, (_, fila) in enumerate(df.iterrows(), start=1):
            uid = uid_de_fila(fila)
            posiciones = fila['posiciones']
            etiqueta = etiqueta_de_fila(fila)
            imagenes = descargar_y_procesar_serie(uid, posiciones)
            if len(imagenes) != len(posiciones):
                raise RuntimeError(f'Cantidad incorrecta de cortes en serie {uid}')
            cantidad = len(imagenes)
            x[cursor:cursor + cantidad] = imagenes
            y[cursor:cursor + cantidad] = etiqueta
            for j, pos in enumerate(posiciones):
                registros.append({'indice_imagen':cursor+j, 'SeriesInstanceUID':uid,
                                  'posicion_relativa':pos, 'etiqueta':etiqueta,
                                  'PatientID':fila.get('PatientID', '')})
            cursor += cantidad
            print(f'{nombre}: serie {i}/{len(df)} | imágenes {cursor}/{total}')
            if eliminar_dicom:
                shutil.rmtree(CARPETA_DICOM / uid, ignore_errors=True)
        x.flush(); y.flush()
        del x, y
        tmp_x.replace(ruta_x)
        tmp_y.replace(ruta_y)
        pd.DataFrame(registros).to_csv(
            CARPETA_RESULTADOS / f'manifest_{sufijo}.csv', index=False)
        print(f'Completado: {ruta_x} ({cursor} imágenes)')
    except Exception:
        del x, y
        tmp_x.unlink(missing_ok=True)
        tmp_y.unlink(missing_ok=True)
        raise


## 8. Configuración de ejecución (descarga apagada por defecto)

- `EJECUTAR_DESCARGA = False`: permite recorrer y revisar el notebook **sin acceso a los DICOM**.
- `MODO_PRUEBA = True`: cuando se habilite la descarga, procesa **una sola serie** del conjunto elegido.
- `MODO_PRUEBA = False`: procesa la totalidad del conjunto elegido (requiere tiempo, disco, conexión y disponibilidad del servicio).
- El procesamiento completo de los seis conjuntos se habilita explícitamente con `TODOS_LOS_CONJUNTOS = True`.

**Precaución:** los escenarios balanceado y desbalanceado pueden contener series repetidas. Este ejemplo guarda cada conjunto por separado y puede requerir espacio considerable. No subas las imágenes DICOM ni archivos `.npy` de gran tamaño al repositorio GitHub.

In [ ]:
EJECUTAR_DESCARGA = False   # CAMBIAR EXPLÍCITAMENTE A True para descargar
MODO_PRUEBA = True         # True = una serie; False = conjunto(s) completo(s)
TODOS_LOS_CONJUNTOS = False
CONJUNTO_ELEGIDO = 'balanceado_test'  # cambiar por cualquiera de NOMBRES
ELIMINAR_DICOM_TRAS_PROCESAR = False

if EJECUTAR_DESCARGA:
    seleccion = NOMBRES if TODOS_LOS_CONJUNTOS else [CONJUNTO_ELEGIDO]
    for nombre in seleccion:
        if nombre not in conjuntos:
            raise KeyError(f'Conjunto desconocido: {nombre}')
        procesar_conjunto(
            nombre, conjuntos[nombre],
            max_series=1 if MODO_PRUEBA else None,
            eliminar_dicom=ELIMINAR_DICOM_TRAS_PROCESAR
        )
else:
    print('Descarga desactivada. El cuaderno contiene el flujo, pero no se descargaron DICOM.')


## 9. Comprobación de salida (solo si se ejecutó la descarga)

Los archivos resultantes son `X_<conjunto>.npy`, `y_<conjunto>.npy` y `manifest_<conjunto>.csv` dentro de `resultados_npy/`. En modo prueba se añade el sufijo `_prueba`.


In [ ]:
archivos = sorted(CARPETA_RESULTADOS.glob('*.npy')) if CARPETA_RESULTADOS.exists() else []
if not archivos:
    print('No hay resultados .npy: la descarga no se ha ejecutado.')
else:
    for ruta in archivos:
        matriz = np.load(ruta, mmap_mode='r')
        print(ruta.name, matriz.shape, matriz.dtype)


### Nota de reproducibilidad

La disponibilidad de una serie depende de TCIA y de que sus CSV contengan o permitan reconstruir **SeriesInstanceUID**. La lectura se ha planteado para volúmenes **DICOM multiframe** con forma `(n_cortes, alto, ancho)`; si alguna serie se almacena en múltiples archivos DICOM de un solo cuadro, será necesario adaptar la lectura y ordenar los cortes mediante sus metadatos antes del procesamiento. El flujo completo con las seis particiones **no se ha ejecutado ni validado aquí**. La normalización es por máximo de intensidad, no se aplica `preprocess_input` de ImageNet. Para cualquier evaluación deben conservarse las particiones por paciente y comprobarse los volúmenes efectivamente procesados.
